# SQL Notebook for SQL Server 2022 data virtualization for delta tables on S3 object storage

This is a SQL notebook to show the data virtualization capabilities in SQL Server 2022 for delta tables on S3 object storage. You can find all the scripts and files referenced in this notebook at [https://aka.ms/sqlserver2022demos](https://aka.ms/sqlserver2022demos) under the datavirt/delta folder

# Prerequisites for the exercise

- SQL Server 2022 Evaluation Edition with the Database Engine and PolyBase Query Service for External Data Feature installed. You can use the defaults in setup for Polybase.
- VM or computer with 2 CPUs and at least 8Gb RAM.
- SQL Server Management Studio (SSMS). The latest 18.x build or 19.x build will work.

    **Note**: The following pre-requisites are for non-Microsoft software. The use of this software does not represent any official endorsement from Microsoft. This software is not supported by Microsoft so any issues using this software are up to the user to resolve.

- The **minio** server for Windows which you can download at https://min.io/download#/windows. For the demo I assume you have created a directory called c:\minio and have downloaded the minio.exe for Windows into that directory.
- openssl for Windows which you can download at https://slproweb.com/products/Win32OpenSSL.html. I chose the Win64 OpenSSL v3.0.5 MSI option.

# Steps to setup minio for the exercise

- Download minio.exe for Windows into c:\minio.exe
- Download the openssl for Windows MSI and run the installer. Use all the defaults.
- Set a system environment variable OPENSSL_CONF=C:\Program Files\OpenSSL-Win64\bin\openssl.cfg and add to the system environment variable PATH c:\Program Files\OpenSSL-Win64\bin
- Generate a private key using the following command from the c:\minio directory.

    `openssl genrsa -out private.key 2048`

- Copy the supplied openssl.conf to c:\minio. Edit openssl.conf by changing IP.2 to your local IP address and DNS.2 to your local computer name.
- From the c:\minio directory run the following command to generate a self-signed certificate

    `openssl req -new -x509 -nodes -days 730 -key private.key -out public.crt -config openssl.conf`

- Double-click the public.crt file and select Install Certificate. Choose Local Machine, Place all certificates in the following store, Browse, and select Trusted Root Certification Authorities.
- Copy the private.key and public.crt files from c:\minio into %%USERPROFILE%%\.minio\certs.

# Steps to use mino for the exercise

From a command prompt at the c:\\minio directory start the minio server with the following command (example syntax for Powershell)

`.\minio.exe server c:\minio --console-address ":9001"`

This program will startup and "run forever" until stopped with +. Your output should look similar to the following (the IP addresses will match your local IP):

```
MinIO Object Storage Server
Copyright: 2015-2022 MinIO, Inc.
License: GNU AGPLv3 <https://www.gnu.org/licenses/agpl-3.0.html>
Version: RELEASE.2022-07-30T05-21-40Z (go1.18.4 windows/amd64)

Status:         1 Online, 0 Offline.
API: https://<local IP>:9000  https://127.0.0.1:9000
RootUser: <user>
RootPass: <password>
Console: https://<local IP>:9001 https://127.0.0.1:9001
RootUser: <user>
RootPass: <password>

Command-line: https://docs.min.io/docs/minio-client-quickstart-guide
   $ mc.exe alias set myminio https://<local IP>:9000 <user> <password>
``
Documentation: https://docs.min.io

```

1. Test your connection and browse the minio storage system using a web browser. Use the address [https://127.0.0.1:9001](https://127.0.0.1:9001/). You should be presented with a login screen. Use the defaut root user and password which are displayed on the minio server screen.
    
2. On the left-hand side menu, click on Identity and Users. Select Create User. Create a user name with password. Select the readwrite policy for the user. This is the user and password you will use for the SECRET value in creates3creds.sql.
    
3. Select menu for Buckets. Select Create Bucket. Use a Bucket Name of **delta**. Leave all defaults and select Create Bucket.
    
4. Select Browse for this new bucket and select Upload folder. Choose the **people-10m** folder from the example files. This will upload all the necessary files for the delta table and create a new folder called people-10m under the delta bucket.

# Steps to use SQL Server 2022 for the exercise

Run each cell one at time to use SQL Server 2022 for the exercise

## Step 1 - Get WideWorldImporters

Copy the **WideWorldImporters** sample database from [https://aka.ms/WideWorldImporters](https://aka.ms/WideWorldImporters) to a local directory (The restore script assumes **c:\\sql\_sample\_databases**)

## Step 2 - Restore WideWorldImporters

Edit the following T-SQL to restore the WideWorldImporters database for your correct files paths and execute the statement

In [4]:
USE master;
GO
DROP DATABASE IF EXISTS WideWorldImporters;
GO
RESTORE DATABASE WideWorldImporters FROM DISK = 'c:\sql_sample_databases\WideWorldImporters-Full.bak' with
MOVE 'WWI_Primary' TO 'c:\sql_sample_databases\WideWorldImporters.mdf',
MOVE 'WWI_UserData' TO 'c:\sql_sample_databases\WideWorldImporters_UserData.ndf',
MOVE 'WWI_Log' TO 'c:\sql_sample_databases\WideWorldImporters.ldf',
MOVE 'WWI_InMemory_Data_1' TO 'c:\sql_sample_databases\WideWorldImporters_InMemory_Data_1',
stats=5;
GO
ALTER DATABASE WideWorldImporters SET QUERY_STORE CLEAR ALL;
GO

Commands completed successfully.

Commands completed successfully.

5 percent processed.

10 percent processed.

15 percent processed.

20 percent processed.

... (105 outputs left out)

100 percent processed.

RESTORE DATABASE successfully processed 58455 pages in 1.801 seconds (253.567 MB/sec).

Commands completed successfully.

Total execution time: 00:00:07.326

# Step 3 - Configure Polybase

Enable polybase configuration options

In [5]:
EXEC sp_configure 'polybase enabled', 1;
GO
RECONFIGURE;
GO
EXEC sp_configure 'allow polybase export', 1;
GO
RECONFIGURE;
GO

Configuration option 'polybase enabled' changed from 1 to 1. Run the RECONFIGURE statement to install.

Commands completed successfully.

Configuration option 'allow polybase export' changed from 1 to 1. Run the RECONFIGURE statement to install.

Commands completed successfully.

Total execution time: 00:00:00.037

# Step 4 - Master Key

Create a master key to protect the database scoped credential. Edit the following statement to put in a strong password

In [7]:
USE [WideWorldImporters]
GO
IF NOT EXISTS (SELECT * FROM sys.symmetric_keys WHERE name = '##MS_DatabaseMasterKey##')
	CREATE MASTER KEY ENCRYPTION BY PASSWORD = '<password>';
GO

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.057

# Step 5 - Database Scoped Credential

Create a database scoped credential. Edit the following statement to put in the user and password from RootUser and RootPwd from minio

In [9]:
USE [WideWorldImporters];
GO
-- Have to drop the data source first if it exists
IF EXISTS (SELECT * FROM sys.external_data_sources WHERE name = 's3_wwi')
	DROP EXTERNAL DATA SOURCE s3_wwi;
IF EXISTS (SELECT * FROM sys.database_scoped_credentials WHERE name = 's3_wwi_cred')
    DROP DATABASE SCOPED CREDENTIAL s3_wwi_cred;
GO
CREATE DATABASE SCOPED CREDENTIAL s3_wwi_cred
WITH IDENTITY = 'S3 Access Key',
SECRET = '<user>:<password>';
GO

Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.008

# Step 6 - External Data Source

Create an external data source. Edit the script to put in your local IP address

In [10]:
USE [WideWorldImporters];
GO
IF EXISTS (SELECT * FROM sys.external_data_sources WHERE name = 's3_wwi')
	DROP EXTERNAL DATA SOURCE s3_wwi;
GO
CREATE EXTERNAL DATA SOURCE s3_wwi
WITH
(
 LOCATION = 's3://<local IP>:9000'
,CREDENTIAL = s3_wwi_cred
);
GO

Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.006

# Step 7 - External file format

Create an external format file for paruqet to be used for external tables

In [11]:
USE [WideWorldImporters];
GO
IF EXISTS (SELECT * FROM sys.external_file_formats WHERE name = 'ParquetFileFormat')
	DROP EXTERNAL FILE FORMAT ParquetFileFormat;
CREATE EXTERNAL FILE FORMAT ParquetFileFormat WITH(FORMAT_TYPE = PARQUET);
GO

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.009

# Step 8 - Query the delta table

Query the delta table under S3 storage using OPENROWSET()

In [1]:
USE [WideWorldImporters];
GO
SELECT * FROM OPENROWSET
(BULK '/delta/people-10m', 
FORMAT = 'DELTA', DATA_SOURCE = 's3_wwi') as [people];
GO

Commands completed successfully.

(10000000 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:59.293

id,firstName,middleName,lastName,gender,birthDate,ssn,salary
3770232,Carla,Leticia,De Angelis,F,1966-04-19 05:00:00.0000000,919-13-8946,72390
3770233,Margit,Marna,Elam,F,1985-03-31 05:00:00.0000000,931-39-1662,81855
3770234,Eric,Concepcion,Parfitt,F,1987-05-10 04:00:00.0000000,963-63-6874,60669
3770235,Margarett,Lahoma,Barwell,F,1982-06-08 04:00:00.0000000,903-10-6577,61307
3770236,Kristy,Eleanora,Zorzetti,F,1977-07-13 04:00:00.0000000,940-73-8455,49728
3770237,Monique,Angella,Dirr,F,1974-12-20 05:00:00.0000000,938-96-1038,53678
3770238,Ronni,Brittaney,O'Moylan,F,1980-10-26 04:00:00.0000000,929-24-4471,110457
3770239,Kristie,Abbie,Posten,F,1961-10-23 04:00:00.0000000,901-61-2111,56105
3770240,Cathy,Lauretta,Slinn,F,1992-08-19 04:00:00.0000000,999-64-4074,66045
3770241,Felipa,Jacquetta,Mapledoram,F,1955-07-27 04:00:00.0000000,977-69-7004,81472


# Step 9 - Query delta table with a filter

Query the delta table by using a WHERE clause for a column that is not a partitioned column

In [2]:
USE [WideWorldImporters];
GO
SELECT * FROM OPENROWSET
(BULK '/delta/people-10m', 
FORMAT = 'DELTA', DATA_SOURCE = 's3_wwi') as [people]
WHERE people.ssn = '992-28-8780';
GO

Commands completed successfully.

Use a collation that enables string predicate pushdown and file elimination on storage layer.

(1 row affected)

Total execution time: 00:00:03.568

id,firstName,middleName,lastName,gender,birthDate,ssn,salary
3766886,Maryrose,Nelly,Tolhurst,F,1968-03-19 05:00:00.0000000,992-28-8780,65554


# Step 10 - Query delta table with a filter using partitioned column

Query the delta table by using a WHERE clause for a column that is a partiitione column. Not the peformance difference.

In [3]:
USE [WideWorldImporters];
GO
SELECT * FROM OPENROWSET
(BULK '/delta/people-10m', 
FORMAT = 'DELTA', DATA_SOURCE = 's3_wwi') as [people]
WHERE [people].id = 10000000;
GO

Commands completed successfully.

(1 row affected)

Total execution time: 00:00:00.347

id,firstName,middleName,lastName,gender,birthDate,ssn,salary
10000000,Joshua,Chas,Broggio,M,1968-07-22 04:00:00.0000000,988-61-6247,85279


# Step 11 - Use CETAS to create parquet files from a delta table

Execute the following T-SQL to create an external table based on parquet files sources from a query from a delta table

In [4]:
USE [WideworldImporters];
GO
IF EXISTS (SELECT * FROM sys.objects WHERE NAME = 'PEOPLE10M_60s')
	DROP EXTERNAL TABLE PEOPLE10M_60s;
GO
CREATE EXTERNAL TABLE PEOPLE10M_60s
WITH 
(   LOCATION = '/delta/1960s',
    DATA_SOURCE = s3_wwi,  
    FILE_FORMAT = ParquetFileFormat)  
AS
SELECT * FROM OPENROWSET
(BULK '/delta/people-10m', FORMAT = 'DELTA', DATA_SOURCE = 's3_wwi') as [people]
WHERE YEAR(people.birthDate) > 1959 AND YEAR(people.birthDate) < 1970;
GO

Commands completed successfully.

Commands completed successfully.

(2080397 rows affected)

Total execution time: 00:00:07.333

# Step 12 - Browse new parquet files

use the minio console to see a new folder called 1960s under the delta bucket with new parquet files

# Step 13 - Query new external table based on parquet

A new folder exists as a subset of the original data. Query the new external table

In [5]:
USE [WideWorldImporters];
GO
SELECT * FROM PEOPLE10M_60s
ORDER BY birthDate;
GO

Commands completed successfully.

(2080397 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:16.529

id,firstName,middleName,lastName,gender,birthDate,ssn,salary
864537,Belkis,Marci,Galer,F,1960-01-08 05:00:00.0000000,973-13-6596,91192
1235984,Daniele,Darla,Fitter,F,1960-01-08 05:00:00.0000000,996-57-5071,48076
9868408,Lynn,Kelvin,Letterick,M,1960-01-08 05:00:00.0000000,943-57-8866,64576
1060672,Yvette,Reiko,Peye,F,1960-01-08 05:00:00.0000000,983-58-4184,68260
9696542,Kimber,Halina,Fannin,F,1960-01-08 05:00:00.0000000,942-89-1684,107463
4963321,Marilu,Season,Piscopiello,F,1960-01-08 05:00:00.0000000,921-85-6976,76928
4362307,Xenia,Carolin,Caldecutt,F,1960-01-08 05:00:00.0000000,980-87-6750,90586
2205786,Simona,Rosio,Flintiff,F,1960-01-08 05:00:00.0000000,950-70-8690,80583
8952371,Lamont,August,Braim,M,1960-01-08 05:00:00.0000000,921-11-9797,88980
4514267,Kaye,Ossie,Knowlys,F,1960-01-08 05:00:00.0000000,931-81-6906,64175
